# crop-mesh-detector on Colab (Stage 1 + Stage 2)

Runs **both stages** of the two-stage pipeline on a free NVIDIA GPU instead of your laptop's
CPU. PlantVillage and PlantDoc (the merged project dataset -- see `src/data/merged.py`) are
fetched straight from their sources on Colab's fast network -- no manual data transfer.

**Stage 1** (`src/train_local.py`): local-only supervised training per node, no knowledge
exchange. Writes checkpoints, manifest, and results to `outputs/stage1_local/`.

**Stage 2** (`src/train_mesh.py`): mesh prototype + probe-logit exchange across all 13
nodes, warm-started from stage 1's saved checkpoints -- never a fresh random init. Writes
checkpoints, results, and a sustainability report to `outputs/stage2_mesh/`. Runs after
stage 1 has finished and been spot-checked in this same session.

If stage 1 already finished in an **earlier** session and you don't want to retrain it,
skip straight to step 8 below and upload `output_mesh_colab/stage1_local/` instead of
running steps 4-7.

**Trains each of the 3 architectures as a separate step, in both stages.** Colab's free tier
caps GPU usage, so training all 3 in one long run risks losing everything to a disconnect
partway through. Instead, each architecture's cell trains, exports a Raspberry-Pi-ready
bundle for *that* architecture, and downloads its results immediately -- so even if the
session disconnects before you get to the next architecture, whatever finished is already
safely on your Mac. Results accumulate into one combined `results_summary.json` per stage
as you go (as long as the session stays connected between cells), so the final comparison +
bulk export still covers all 3.

**Before running anything**: `Runtime` menu -> `Change runtime type` -> Hardware accelerator -> `T4 GPU` -> Save.


In [ ]:
# Confirm the GPU runtime is actually attached
!nvidia-smi

## 1. Clone the project (private repo)

This repo is private, so cloning it here needs a GitHub personal access token --
generate one at github.com -> Settings -> Developer settings -> Personal access tokens
(fine-grained, read-only access to this one repo is enough). `getpass` keeps it out of
the notebook's saved output/history.

Clones the `kb-tuning-02` branch specifically -- `src/train_local.py`/`src/train_mesh.py`
(the two-stage pipeline this notebook runs) only exist there, not on `main` yet.


In [ ]:
from getpass import getpass

token = getpass('GitHub personal access token: ')
!git clone --branch kb-tuning-02 https://{token}@github.com/karboon1008/crop-mesh-detector.git /content/crop-mesh-detector
del token
%cd /content/crop-mesh-detector
!ls

## 2. Install dependencies

Colab already ships recent `torch`/`tensorflow`/`numpy`; this adds what's missing
(`timm`, `codecarbon`, `onnx`, `onnxruntime`) and pins the rest per `requirements.txt`.


In [ ]:
!pip install -q -r requirements.txt
!pip install -q "protobuf>=5.29.1,<6"

### Excluded nodes

Nodes 0, 3, 4, 6, 7, and 12 are left out of the combined stage-1/stage-2 archives
downloaded below (per-architecture bundles never included checkpoints in the first
place). The checkpoints still exist on the Colab VM and are still used for
training/export/comparison there -- this only trims what leaves the VM.


In [ ]:
import shutil
from pathlib import Path

EXCLUDED_NODES = {"node_0", "node_3", "node_4", "node_6", "node_7", "node_12"}

def archive_excluding_nodes(base_name, src_dir):
    """Like shutil.make_archive(base_name, 'zip', src_dir), but drops EXCLUDED_NODES'
    checkpoint files from the archive -- the originals under src_dir are untouched."""
    def skip_excluded_nodes(dir_path, names):
        if Path(dir_path).name == "checkpoints" or Path(dir_path).parent.name == "checkpoints":
            return [n for n in names if Path(n).stem in EXCLUDED_NODES]
        return []

    tmp_dir = "/content/_archive_tmp"
    shutil.rmtree(tmp_dir, ignore_errors=True)
    shutil.copytree(src_dir, tmp_dir, ignore=skip_excluded_nodes)
    shutil.make_archive(base_name, "zip", tmp_dir)
    shutil.rmtree(tmp_dir)

## 3. Fetch PlantVillage + PlantDoc

Same scripts as local. Stage 1 trains on the merged PlantVillage+PlantDoc project dataset
(`src/data/merged.py`), so both need to be present. PlantVillage comes via the
`tensorflow_datasets` library catalog entry (no Kaggle account needed) and is written out as
`.jpg` files under `data/PlantVillage/`; PlantDoc is fetched from its GitHub release into
`data/PlantDoc/`. Takes a few minutes on Colab's network. Shared by all 3 architectures
below -- only needs to run once per session.


In [ ]:
!python scripts/download_plantvillage.py
!python scripts/download_plantdoc.py

## 4. Model 1/3: mobilenet_v3_small (stage 1: local-only)

`--arch` overrides `config.yaml`'s architecture list for this one run. `src/train_local.py`
picks `cuda` automatically when available -- on Colab's GPU runtime that resolves to
the T4, no code change needed. Results merge into `outputs/stage1_local/results_summary.json`
(`--fresh` here means: start a brand-new stage-1 run, discarding any old results/manifest
from a previous session).


In [ ]:
!python -m src.train_local --config config.yaml --arch mobilenet_v3_small --fresh

In [ ]:
# Export this architecture's best node to a Pi-ready bundle, then download its
# results immediately -- protects this model's results even if the next one fails.
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage1_local/checkpoints --results outputs/stage1_local/results_summary.json --arch mobilenet_v3_small --node node_0 --output-dir outputs/pi_export/mobilenet_v3_small

from google.colab import files

shutil.make_archive('/content/mobilenet_v3_small_bundle', 'zip', 'outputs/pi_export/mobilenet_v3_small')
files.download('/content/mobilenet_v3_small_bundle.zip')

## 5. Model 2/3: efficientnet_lite0

No `--fresh` this time -- merges into the same `outputs/stage1_local/results_summary.json` from step 4.


In [ ]:
!python -m src.train_local --config config.yaml --arch efficientnet_lite0

In [ ]:
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage1_local/checkpoints --results outputs/stage1_local/results_summary.json --arch efficientnet_lite0 --node node_0 --output-dir outputs/pi_export/efficientnet_lite0

shutil.make_archive('/content/efficientnet_lite0_bundle', 'zip', 'outputs/pi_export/efficientnet_lite0')
files.download('/content/efficientnet_lite0_bundle.zip')

## 6. Model 3/3: mobilevit_xxs


In [ ]:
!python -m src.train_local --config config.yaml --arch mobilevit_xxs

In [ ]:
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage1_local/checkpoints --results outputs/stage1_local/results_summary.json --arch mobilevit_xxs --node node_0 --output-dir outputs/pi_export/mobilevit_xxs

shutil.make_archive('/content/mobilevit_xxs_bundle', 'zip', 'outputs/pi_export/mobilevit_xxs')
files.download('/content/mobilevit_xxs_bundle.zip')

## 7. Compare all 3 and download the combined stage-1 results

Only meaningful if the session stayed connected through steps 4-6 (so
`outputs/stage1_local/results_summary.json` actually has all 3 architectures merged in).
`--all` re-exports every architecture's best node into its own subfolder -- redundant with
steps 4/5/6 above, but convenient as one combined bundle covering all 3 to compare on the Pi.

Stage 1 alone has no sustainability report (that's produced by stage 2, which combines
compute + communication cost) -- `run_state.json` here has stage 1's accumulated compute
energy/duration instead. Once you've looked over these results, continue to stage 2 below --
it warm-starts from `outputs/stage1_local/checkpoints/` still sitting in this session, no
need to re-upload anything.


In [ ]:
!python scripts/export_for_pi.py --all --checkpoints-dir outputs/stage1_local/checkpoints --results outputs/stage1_local/results_summary.json
!cat outputs/stage1_local/results_summary.json
!cat outputs/stage1_local/run_state.json

In [ ]:
archive_excluding_nodes('/content/stage1_local', 'outputs/stage1_local')
files.download('/content/stage1_local.zip')

## 8. (Alternative) Resume from an existing stage-1 run

**Skip this if you just ran steps 4-7 above in this same session** -- stage 2 already has
what it needs. This step is only for resuming in a **fresh** session where stage 1 already
finished earlier and its output is sitting locally at `output_mesh_colab/stage1_local/`
rather than in this VM.

Zip that folder's *contents* (not the folder itself, so the paths inside match what steps
4-7 above produce) on your Mac:

```
cd output_mesh_colab/stage1_local
zip -r ../../stage1_local_upload.zip .
cd ../..
```

Then run the cell below and pick `stage1_local_upload.zip` when prompted. It's unpacked
into `outputs/stage1_local/`, which is exactly where `src.train_mesh` (and its stage-1
manifest check) expect to find it.


In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))

Path('outputs/stage1_local').mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(zip_name, 'outputs/stage1_local')
!ls outputs/stage1_local
!cat outputs/stage1_local/manifest.json

## 9. Model 1/3: mobilenet_v3_small (stage 2: mesh exchange)

`src/train_mesh.py` continues from this architecture's stage-1 checkpoint -- whether it
was trained in this session (steps 4-7) or uploaded from a previous one (step 8). It
rebuilds the same dataset split as stage 1 and checks it against stage 1's manifest before
trusting the checkpoint, so a config drifted between stages fails loudly instead of
silently pairing a checkpoint with the wrong node's data. Results merge into
`outputs/stage2_mesh/results_summary.json` (`--fresh` here means: start a brand-new
stage-2 run, discarding any old results/manifest from a previous session).


In [ ]:
!python -m src.train_mesh --config config.yaml --arch mobilenet_v3_small --fresh

In [ ]:
# Export this architecture's best mesh-eval node to a Pi-ready bundle, then download its
# results immediately -- protects this model's results even if the next one fails.
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage2_mesh/checkpoints --results outputs/stage2_mesh/results_summary.json --arch mobilenet_v3_small --node node_0 --output-dir outputs/pi_export_stage2/mobilenet_v3_small

shutil.make_archive('/content/mobilenet_v3_small_stage2_bundle', 'zip', 'outputs/pi_export_stage2/mobilenet_v3_small')
files.download('/content/mobilenet_v3_small_stage2_bundle.zip')

## 10. Model 2/3: efficientnet_lite0 (stage 2: mesh exchange)

No `--fresh` this time -- merges into the same `outputs/stage2_mesh/results_summary.json` from step 9.


In [ ]:
!python -m src.train_mesh --config config.yaml --arch efficientnet_lite0

In [ ]:
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage2_mesh/checkpoints --results outputs/stage2_mesh/results_summary.json --arch efficientnet_lite0 --node node_0 --output-dir outputs/pi_export_stage2/efficientnet_lite0

shutil.make_archive('/content/efficientnet_lite0_stage2_bundle', 'zip', 'outputs/pi_export_stage2/efficientnet_lite0')
files.download('/content/efficientnet_lite0_stage2_bundle.zip')

## 11. Model 3/3: mobilevit_xxs (stage 2: mesh exchange)


In [ ]:
!python -m src.train_mesh --config config.yaml --arch mobilevit_xxs

In [ ]:
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage2_mesh/checkpoints --results outputs/stage2_mesh/results_summary.json --arch mobilevit_xxs --node node_0 --output-dir outputs/pi_export_stage2/mobilevit_xxs

shutil.make_archive('/content/mobilevit_xxs_stage2_bundle', 'zip', 'outputs/pi_export_stage2/mobilevit_xxs')
files.download('/content/mobilevit_xxs_stage2_bundle.zip')

## 12. Compare all 3 and download the combined stage-2 results

Only meaningful if the session stayed connected through steps 9-11 (so
`outputs/stage2_mesh/results_summary.json` actually has all 3 architectures merged in).
`--all` re-exports every architecture's best mesh-eval node into its own subfolder --
redundant with steps 9/10/11 above, but convenient as one combined bundle covering all 3 to
compare on the Pi.

`run_state.json` and `sustainability_report/` here cover stage 2's accumulated compute +
communication cost across all rounds and architectures trained this session -- the
collaboration-gain numbers (mesh vs. local-only accuracy) live in `results_summary.json`.


In [ ]:
!python scripts/export_for_pi.py --all --checkpoints-dir outputs/stage2_mesh/checkpoints --results outputs/stage2_mesh/results_summary.json
!cat outputs/stage2_mesh/results_summary.json
!cat outputs/stage2_mesh/run_state.json

In [ ]:
archive_excluding_nodes('/content/stage2_mesh', 'outputs/stage2_mesh')
files.download('/content/stage2_mesh.zip')